![MuJoCo banner](https://raw.githubusercontent.com/google-deepmind/mujoco/main/banner.png)


### Copyright notice


> <p><small><small>Copyright 2025 DeepMind Technologies Limited.</small></p>
> <p><small><small>Licensed under the Apache License, Version 2.0 (the "License"); you may not use this file except in compliance with the License. You may obtain a copy of the License at <a href="http://www.apache.org/licenses/LICENSE-2.0">http://www.apache.org/licenses/LICENSE-2.0</a>.</small></small></p>
> <p><small><small>Unless required by applicable law or agreed to in writing, software distributed under the License is distributed on an "AS IS" BASIS, WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied. See the License for the specific language governing permissions and limitations under the License.</small></small></p>


# T1 Humanoid Locomotion with Force Sensors <a href="https://colab.research.google.com/github/google-deepmind/mujoco_playground/blob/main/learning/notebooks/t1_joystick_flat_terrain.ipynb"><img src="https://colab.research.google.com/assets/colab-badge.svg" width="140" align="center"/></a>

In this notebook, we'll train and evaluate the T1 humanoid robot with force sensors on flat terrain. The T1 robot is equipped with force sensors on both feet to provide additional tactile feedback for improved balance and locomotion control.

**A Colab runtime with GPU acceleration is required.** If you're using a CPU-only runtime, you can switch using the menu "Runtime > Change runtime type".


In [ ]:
#@title Install pre-requisites
!pip install mujoco
!pip install mujoco_mjx
!pip install brax


In [ ]:
#@title Import packages
import jax
import jax.numpy as jp
import numpy as np
import matplotlib.pyplot as plt
from mujoco_playground import registry
from mujoco_playground.config import locomotion_params
from brax.training.agents.ppo import networks as ppo_networks
from brax.training.agents.ppo import train as ppo
import functools
from datetime import datetime
from IPython.display import clear_output, display
import mujoco
from mujoco import mjx


# T1 Humanoid Robot with Force Sensors

The T1 is a humanoid robot with force sensors on both feet. These sensors provide:
- **Ground reaction force measurements** (3D force vectors)
- **Enhanced balance control** through tactile feedback
- **Improved terrain adaptation** capabilities
- **Better gait stability** and foot placement

Let's load the T1 environment and examine its configuration:


In [ ]:
# Load T1 Joystick environment for flat terrain
env_name = 'T1JoystickFlatTerrain'
env = registry.load(env_name)
env_cfg = registry.get_default_config(env_name)

print(f"Environment: {env_name}")
print(f"Action size: {env.action_size}")
print(f"Observation space:")
print(f"  State: {env.observation_size}")
print(f"  Privileged state: {env.privileged_observation_size}")


## Force Sensor Integration

The T1 robot is equipped with force sensors on both feet. Let's examine the sensor data:


In [ ]:
# Test force sensors
rng = jax.random.PRNGKey(42)
state = env.reset(rng)

# Get force sensor readings
left_force = env.get_left_foot_force(state.data)
right_force = env.get_right_foot_force(state.data)
feet_forces = env.get_feet_forces(state.data)

print("Force Sensor Readings:")
print("=" * 30)
print(f"Left foot force: {left_force} (shape: {left_force.shape})")
print(f"Right foot force: {right_force} (shape: {right_force.shape})")
print(f"Combined feet forces: {feet_forces} (shape: {feet_forces.shape})")
print(f"\nForce components: [Fx, Fy, Fz] (Newtons)")
print(f"Left foot:  Fx={left_force[0]:.2f}, Fy={left_force[1]:.2f}, Fz={left_force[2]:.2f}")
print(f"Right foot: Fx={right_force[0]:.2f}, Fy={right_force[1]:.2f}, Fz={right_force[2]:.2f}")


## Training Configuration

Let's set up the training parameters for the T1 robot:


In [ ]:
# Get PPO training parameters
ppo_params = locomotion_params.brax_ppo_config(env_name)

print("PPO Training Parameters:")
print("=" * 30)
for key, value in ppo_params.items():
    if isinstance(value, dict):
        print(f"{key}:")
        for subkey, subvalue in value.items():
            print(f"  {subkey}: {subvalue}")
    else:
        print(f"{key}: {value}")


## Training the Policy

Now let's train the T1 joystick policy with force sensor feedback:


In [ ]:
# Setup training progress tracking
x_data, y_data, y_dataerr = [], [], []
times = [datetime.now()]

def progress(num_steps, metrics):
  clear_output(wait=True)

  times.append(datetime.now())
  x_data.append(num_steps)
  y_data.append(metrics["eval/episode_reward"])
  y_dataerr.append(metrics["eval/episode_reward_std"])

  plt.xlim([0, ppo_params["num_timesteps"] * 1.25])
  plt.xlabel("# environment steps")
  plt.ylabel("reward per episode")
  plt.title(f"T1 Training Progress - Reward: {y_data[-1]:.3f}")
  plt.errorbar(x_data, y_data, yerr=y_dataerr, color="blue")

  display(plt.gcf())

# Setup training function
randomizer = registry.get_domain_randomizer(env_name)
ppo_training_params = dict(ppo_params)
network_factory = ppo_networks.make_ppo_networks
if "network_factory" in ppo_params:
  del ppo_training_params["network_factory"]
  network_factory = functools.partial(
      ppo_networks.make_ppo_networks,
      **ppo_params.network_factory
  )

train_fn = functools.partial(
    ppo.train, **dict(ppo_training_params),
    network_factory=network_factory,
    randomization_fn=randomizer,
    progress_fn=progress
)

print("Training setup complete. Ready to train T1 with force sensors!")


In [ ]:
# Train the policy
print("Starting T1 training with force sensors...")
print("This may take several minutes depending on your hardware.")

from mujoco_playground import wrapper

make_inference_fn, params, metrics = train_fn(
    environment=env,
    eval_env=registry.load(env_name, config=env_cfg),
    wrap_env_fn=wrapper.wrap_for_brax_training,
)

print(f"\nTraining completed!")
print(f"Time to JIT: {times[1] - times[0]}")
print(f"Time to train: {times[-1] - times[1]}")
print(f"Final reward: {y_data[-1]:.3f} ± {y_dataerr[-1]:.3f}")


## Policy Evaluation and Force Sensor Analysis

Let's evaluate the trained policy and analyze the force sensor data:


In [ ]:
# Setup evaluation environment
env_cfg = registry.get_default_config(env_name)
env_cfg.pert_config.enable = True
env_cfg.pert_config.velocity_kick = [3.0, 6.0]
env_cfg.pert_config.kick_wait_times = [5.0, 15.0]
env_cfg.command_config.a = [1.5, 0.8, 2*jp.pi]
eval_env = registry.load(env_name, config=env_cfg)

# JIT compile functions for faster execution
jit_reset = jax.jit(eval_env.reset)
jit_step = jax.jit(eval_env.step)
jit_inference_fn = jax.jit(make_inference_fn(params, deterministic=True))

print("Evaluation environment setup complete.")


In [ ]:
#@title Rollout and Force Sensor Analysis
from mujoco_playground._src.gait import draw_joystick_command

x_vel = 0.0  #@param {type: "number"}
y_vel = 0.0  #@param {type: "number"}
yaw_vel = 3.14  #@param {type: "number"}

def sample_pert(rng):
  rng, key1, key2 = jax.random.split(rng, 3)
  pert_mag = jax.random.uniform(
      key1, minval=0.0, maxval=0.0  # Disable velocity kick for now
  )
  duration_seconds = jax.random.uniform(
      key2, minval=0.05, maxval=0.2
  )
  duration_steps = jp.round(duration_seconds / eval_env.dt).astype(jp.int32)
  state.info["pert_mag"] = pert_mag
  state.info["pert_duration"] = duration_steps
  state.info["pert_duration_seconds"] = duration_seconds
  return rng

rng = jax.random.PRNGKey(0)
rollout = []
modify_scene_fns = []

# Data collection for analysis
swing_peak = []
rewards = []
linvel = []
angvel = []
track = []
foot_vel = []
rews = []
contact = []
force_data = []  # Store force sensor data

command = jp.array([x_vel, y_vel, yaw_vel])

state = jit_reset(rng)
if state.info["steps_since_last_pert"] < state.info["steps_until_next_pert"]:
  rng = sample_pert(rng)
state.info["command"] = command

for i in range(env_cfg.episode_length):
  if state.info["steps_since_last_pert"] < state.info["steps_until_next_pert"]:
    rng = sample_pert(rng)
  act_rng, rng = jax.random.split(rng)
  ctrl, _ = jit_inference_fn(state.obs, act_rng)
  state = jit_step(state, ctrl)
  state.info["command"] = command
  
  # Collect data
  rews.append(
      {k: v for k, v in state.metrics.items() if k.startswith("reward/")}
  )
  rollout.append(state)
  swing_peak.append(state.info["swing_peak"])
  rewards.append(
      {k[7:]: v for k, v in state.metrics.items() if k.startswith("reward/")}
  )
  linvel.append(eval_env.get_global_linvel(state.data))
  angvel.append(eval_env.get_gyro(state.data))
  track.append(
      eval_env._reward_tracking_lin_vel(
          state.info["command"], eval_env.get_local_linvel(state.data)
      )
  )

  # Collect force sensor data
  feet_forces = eval_env.get_feet_forces(state.data)
  force_data.append(feet_forces)

  feet_vel = state.data.sensordata[eval_env._foot_linvel_sensor_adr]
  vel_xy = feet_vel[..., :2]
  vel_norm = jp.sqrt(jp.linalg.norm(vel_xy, axis=-1))
  foot_vel.append(vel_norm)

  contact.append(state.info["last_contact"])

  # Visualization
  xyz = np.array(state.data.xpos[eval_env._torso_body_id])
  xyz += np.array([0, 0, 0.2])
  x_axis = state.data.xmat[eval_env._torso_body_id, 0]
  yaw = -np.arctan2(x_axis[1], x_axis[0])
  modify_scene_fns.append(
      functools.partial(
          draw_joystick_command,
          cmd=state.info["command"],
          xyz=xyz,
          theta=yaw,
          scl=abs(state.info["command"][0]) / env_cfg.command_config.a[0],
      )
  )

render_every = 2
fps = 1.0 / eval_env.dt / render_every
traj = rollout[::render_every]
mod_fns = modify_scene_fns[::render_every]

scene_option = mujoco.MjvOption()
scene_option.geomgroup[2] = True
scene_option.geomgroup[3] = False
scene_option.flags[mujoco.mjtVisFlag.mjVIS_CONTACTPOINT] = True
scene_option.flags[mujoco.mjtVisFlag.mjVIS_TRANSPARENT] = False
scene_option.flags[mujoco.mjtVisFlag.mjVIS_PERTFORCE] = True

frames = eval_env.render(
    traj,
    camera="track",
    scene_option=scene_option,
    width=640,
    height=480,
    modify_scene_fns=mod_fns,
)

# Show video (if mediapy is available)
try:
    import mediapy as media
    media.show_video(frames, fps=fps, loop=False)
except ImportError:
    print("Video rendering requires mediapy. Install with: pip install mediapy")
    print(f"Generated {len(frames)} frames at {fps:.1f} FPS")


## Force Sensor Analysis

Let's analyze the force sensor data to understand how the robot uses tactile feedback:


In [ ]:
# Plot force sensor data
force_data = jp.array(force_data)
time_steps = jp.arange(len(force_data)) * eval_env.dt

fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# Left foot forces
axes[0, 0].plot(time_steps, force_data[:, 0], 'r-', label='Fx')
axes[0, 0].plot(time_steps, force_data[:, 1], 'g-', label='Fy')
axes[0, 0].plot(time_steps, force_data[:, 2], 'b-', label='Fz')
axes[0, 0].set_title('Left Foot Forces')
axes[0, 0].set_xlabel('Time (s)')
axes[0, 0].set_ylabel('Force (N)')
axes[0, 0].legend()
axes[0, 0].grid(True)

# Right foot forces
axes[0, 1].plot(time_steps, force_data[:, 3], 'r-', label='Fx')
axes[0, 1].plot(time_steps, force_data[:, 4], 'g-', label='Fy')
axes[0, 1].plot(time_steps, force_data[:, 5], 'b-', label='Fz')
axes[0, 1].set_title('Right Foot Forces')
axes[0, 1].set_xlabel('Time (s)')
axes[0, 1].set_ylabel('Force (N)')
axes[0, 1].legend()
axes[0, 1].grid(True)

# Vertical forces comparison
axes[1, 0].plot(time_steps, force_data[:, 2], 'r-', label='Left Fz')
axes[1, 0].plot(time_steps, force_data[:, 5], 'b-', label='Right Fz')
axes[1, 0].set_title('Vertical Forces (Ground Reaction)')
axes[1, 0].set_xlabel('Time (s)')
axes[1, 0].set_ylabel('Force (N)')
axes[1, 0].legend()
axes[1, 0].grid(True)

# Force magnitude
left_mag = jp.sqrt(jp.sum(force_data[:, :3]**2, axis=1))
right_mag = jp.sqrt(jp.sum(force_data[:, 3:]**2, axis=1))
axes[1, 1].plot(time_steps, left_mag, 'r-', label='Left |F|')
axes[1, 1].plot(time_steps, right_mag, 'b-', label='Right |F|')
axes[1, 1].set_title('Force Magnitude')
axes[1, 1].set_xlabel('Time (s)')
axes[1, 1].set_ylabel('|Force| (N)')
axes[1, 1].legend()
axes[1, 1].grid(True)

plt.tight_layout()
plt.show()

# Print force statistics
print("Force Sensor Statistics:")
print("=" * 30)
print(f"Left foot - Max Fz: {jp.max(force_data[:, 2]):.2f} N")
print(f"Left foot - Mean Fz: {jp.mean(force_data[:, 2]):.2f} N")
print(f"Right foot - Max Fz: {jp.max(force_data[:, 5]):.2f} N")
print(f"Right foot - Mean Fz: {jp.mean(force_data[:, 5]):.2f} N")
print(f"Total weight support: {jp.mean(force_data[:, 2] + force_data[:, 5]):.2f} N")


## Conclusion

This notebook demonstrated the T1 humanoid robot with integrated force sensors:

### Key Features:
- **Force Sensors**: 3D force measurement on both feet
- **Enhanced Control**: Improved balance and terrain adaptation
- **Real-time Feedback**: Force data integrated into observations
- **Robust Training**: Domain randomization for sim-to-real transfer

### Benefits of Force Sensors:
1. **Better Balance**: Real-time ground reaction force feedback
2. **Terrain Adaptation**: Surface property detection
3. **Gait Optimization**: Improved foot placement and timing
4. **Research Capabilities**: Enables studies on tactile locomotion

The force sensors provide valuable tactile feedback that enhances the robot's ability to maintain balance and adapt to different terrains, making it more robust and capable for real-world applications.
